# Lezione 6 — Progetto finale: analisi recensioni + RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccasadei-maggioli/corso-nlp-genai-2026/blob/main/lezione_6_progetto_finale_rag/notebook_06_progetto_rag.ipynb)

Eccoci all'**ultima lezione**! 🎉 È il momento di **mettere insieme tutto** ciò che
abbiamo imparato e costruire una vera applicazione.

Ripercorriamo il viaggio fatto finora:
- **L1 — Introduzione & Setup:** ambiente Colab, GPU T4, prima `pipeline` di sentiment.
- **L2 — Embeddings:** trasformare il testo in vettori e fare **ricerca semantica**
  (con `intfloat/multilingual-e5-base`).
- **L3 — Modelli task-specific:** **sentiment**, **NER** ed estrazione di temi.
- **L4 — LLM generativi:** **Qwen2.5-7B-Instruct** in 4-bit sulla T4, prompting, Q&A.
- **L5 — LangChain:** *prompt template*, **catene** (LCEL) e parser per orchestrare l'LLM.

Oggi uniamo questi mattoni in un'unica app: un sistema **RAG** (Retrieval-Augmented
Generation) che **risponde a domande sulle recensioni citando le fonti**.

In questa lezione:
1. capiamo *cos'è* il **RAG** e perché serve (l'architettura ingest → … → generate);
2. **(A)** arricchiamo le recensioni con il **sentiment** (richiamo L3);
3. **(B)** costruiamo il **vector store FAISS** con gli embeddings (richiamo L2);
4. **(C)** avvolgiamo **Qwen** in LangChain come `ChatHuggingFace` (richiamo L4/L5);
5. **(D)** componiamo la **catena RAG con citazioni** e la interroghiamo;
6. **(E)** produciamo un **cruscotto** riassuntivo "business" con pandas.

> 🎯 **Filo conduttore (gran finale):** sempre le stesse **recensioni clienti in
> italiano**. Oggi non aggiungiamo un mattone nuovo: li **assembliamo tutti** in
> un'applicazione completa.

---
### ⚙️ Reminder: attiva la GPU T4
Menu **`Runtime` → `Change runtime type` → Hardware accelerator: `T4 GPU` → `Save`**.
Oggi carichiamo sia il modello di embeddings sia l'LLM 7B: la **GPU è indispensabile**.

> ⏳ **AVVISO sui tempi (utile se stai registrando):** caricare l'LLM 7B la prima volta
> richiede **1–3 minuti** (download) e ogni risposta del RAG può richiedere **qualche
> secondo**. È normale. Se la T4 risulta lenta, più avanti trovi l'alternativa
> **Qwen2.5-3B**.

## 1. Installiamo le librerie

Questa lezione è la "somma" delle precedenti, quindi installiamo tutto ciò che serve in
un colpo solo:
- `transformers`, `accelerate`, `bitsandbytes` → l'**LLM Qwen in 4-bit** (L4);
- `sentence-transformers` → il modello di **embeddings** (L2);
- `faiss-cpu` → il **vector store** per la ricerca dei vettori;
- `langchain`, `langchain-huggingface`, `langchain-community` → l'**orchestrazione** (L5).

> 💡 Come sempre, ogni notebook installa da sé ciò che gli serve, così puoi aprire questa
> lezione in modo indipendente.

In [ ]:
# -q = silenzioso. Essendo molte librerie, la prima installazione richiede 1-2 minuti.
!pip install -q "transformers>=4.45" "accelerate>=0.34" "bitsandbytes>=0.44" "sentence-transformers>=3.0" "faiss-cpu>=1.8" "langchain>=0.3" "langchain-huggingface>=0.1" "langchain-community>=0.3" "sentencepiece>=0.2" "protobuf>=4.0"
print("Librerie installate ✅")

In [ ]:
import torch

print("Versione PyTorch:", torch.__version__)
print("GPU disponibile:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Scheda:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU non attiva. Vai su Runtime > Change runtime type > T4 GPU,")
    print("    altrimenti l'LLM 7B NON si caricherà.")

## 2. Che cos'è il RAG? 🧠

Nella Lezione 4 abbiamo visto un limite degli LLM: sanno solo ciò che hanno "letto"
durante l'addestramento. **Non conoscono i nostri dati** — le nostre recensioni non sono
nel modello. E se gli chiediamo qualcosa che non sa, spesso **inventa** una risposta
plausibile ma falsa: è il fenomeno delle **allucinazioni**.

Il **RAG (Retrieval-Augmented Generation)** risolve esattamente questo problema. L'idea è
semplice: **prima di rispondere**, andiamo a *recuperare* (retrieval) dal nostro archivio
i pezzi di testo più pertinenti alla domanda, e li **forniamo all'LLM come contesto**.
Così l'LLM non risponde "a memoria", ma **leggendo i nostri documenti**.

> 🔑 In una frase: **RAG = ricerca semantica (L2) + LLM generativo (L4)**. La ricerca
> trova i documenti giusti; l'LLM li legge e formula la risposta in linguaggio naturale.

**Perché serve:**
- **riduce le allucinazioni** — l'LLM risponde sui *nostri* dati, non a fantasia;
- **dà risposte aggiornate** — basta aggiornare l'archivio, non riaddestrare il modello;
- **è verificabile** — possiamo far **citare le fonti**, così l'utente controlla.

## 3. L'architettura RAG 🏗️

Un sistema RAG si divide in due fasi.

**Fase di indicizzazione (offline, una volta sola):**

```
ingest  →  chunk  →  embed  →  store
```
- **ingest** — carichiamo i documenti (qui: le recensioni);
- **chunk** — li spezziamo in pezzi ("chunk") di dimensione gestibile;
- **embed** — calcoliamo l'embedding di ogni chunk (L2);
- **store** — li salviamo in un **vector store** (qui: **FAISS**), pronti per la ricerca.

**Fase di interrogazione (online, a ogni domanda):**

```
retrieve  →  generate
```
- **retrieve** — la domanda diventa un embedding e cerchiamo i chunk più simili;
- **generate** — passiamo domanda + chunk recuperati all'LLM, che genera la risposta.

### Una nota sul *chunking*
In generale i documenti lunghi (es. un PDF di 50 pagine) vanno **spezzati**: l'LLM ha un
contesto limitato e la ricerca è più precisa su pezzi piccoli. Ma le **nostre recensioni
sono già brevi** (poche frasi): qui adottiamo la regola più semplice possibile —
**1 documento = 1 recensione**, nessuno split. È la scelta giusta per testi corti come i
nostri; teniamo a mente che con documenti lunghi servirebbe un vero *text splitter*.

### 🖼️ Schema dell'architettura RAG

![Architettura RAG](https://raw.githubusercontent.com/ccasadei-maggioli/corso-nlp-genai-2026/main/assets/rag_architettura.png)

- **Indicizzazione** (offline, una volta): ingest recensioni → *chunk* (1 doc = 1 recensione)
  → *embed* (e5-base) → *store* (FAISS).
- **Interrogazione** (a ogni domanda): domanda utente → *retrieve* (top-k simili dal FAISS)
  → *generate* (LLM Qwen) → risposta **con citazioni**.

## 4. Il nostro dataset: le recensioni 🛒

Ricarichiamo le solite recensioni sintetiche in italiano (schema:
`id, data, prodotto, categoria, rating, titolo, testo`). Sono *riproducibili* (seed fisso)
e non richiedono download esterni. Saranno la **base di conoscenza** del nostro RAG.

In [ ]:
import os, torch
import pandas as pd

# Scarica lo script generatore se non è già nella sessione Colab.
if not os.path.exists("genera_recensioni.py"):
    !wget -q https://raw.githubusercontent.com/ccasadei-maggioli/corso-nlp-genai-2026/main/dati/genera_recensioni.py

import genera_recensioni

df = pd.DataFrame(genera_recensioni.genera_recensioni(n=200, seed=42))
print("Numero di recensioni:", len(df))
df.head(3)

## (A) Arricchimento sintetico: il sentiment 🏷️

Prima di indicizzare, **arricchiamo** le recensioni con metadati utili. Riprendiamo dalla
**Lezione 3** il calcolo del **sentiment** con il modello italiano
`neuraly/bert-base-italian-cased-sentiment`, usando una `pipeline` di Hugging Face. Aggiungiamo
una colonna `sentiment` (`negative` / `neutral` / `positive`) a ogni recensione.

Questo arricchimento ci servirà a due cose: il **cruscotto** finale (sezione E) e i
**filtri** del retriever (esercizio).

> 💡 Allo **stesso modo** potremmo integrare gli altri risultati della L3 — ad esempio le
> **entità (NER)** o i **temi/categorie** — aggiungendo altre colonne. Per tenere il
> progetto compatto qui ci fermiamo al sentiment, ma il pattern è identico.

In [ ]:
from transformers import pipeline

# device=0 -> GPU; -1 -> CPU. Questo modello è piccolo e veloce.
device = 0 if torch.cuda.is_available() else -1

classificatore_sentiment = pipeline(
    task="text-classification",
    model="neuraly/bert-base-italian-cased-sentiment",  # etichette: negative/neutral/positive
    device=device,
)

# Applichiamo il modello a TUTTE le recensioni e aggiungiamo la colonna 'sentiment'.
predizioni = classificatore_sentiment(df["testo"].tolist(), batch_size=16, truncation=True)
df["sentiment"] = [p["label"] for p in predizioni]

print("Distribuzione del sentiment:")
print(df["sentiment"].value_counts())
df[["rating", "prodotto", "sentiment", "testo"]].head(5)

## (B) Embeddings + Vector store FAISS 🔢

Ora la parte **`embed` → `store`** dell'architettura. Nella Lezione 2 abbiamo calcolato gli
embeddings "a mano" con `sentence-transformers`; qui li avvolgiamo nei componenti di
**LangChain**, così si incastrano direttamente nella catena RAG:

- **`HuggingFaceEmbeddings`** — wrapper LangChain attorno al nostro modello
  `intfloat/multilingual-e5-base` (lo stesso della L2);
- **`Document`** — l'unità di LangChain: un testo (`page_content`) + dei **metadati**
  (qui: `id`, `prodotto`, `rating`). I metadati ci serviranno per **citare le fonti** e
  per i **filtri**;
- **`FAISS`** — il **vector store**: indicizza i vettori e fa la ricerca per similarità in
  modo efficiente;
- **`retriever`** — l'oggetto che, data una domanda, restituisce i `k` documenti più
  pertinenti. È il "retrieve" del RAG.

Seguendo la regola **1 documento = 1 recensione**, creiamo un `Document` per recensione.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# Modello di embeddings (lo stesso della L2). normalize_embeddings=True -> coseno.
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    encode_kwargs={"normalize_embeddings": True},
)

# 1 documento = 1 recensione. Nei metadati mettiamo ciò che ci serve per le citazioni.
documenti = [
    Document(
        page_content=r.testo,
        metadata={"id": int(r.id), "prodotto": r.prodotto, "rating": int(r.rating)},
    )
    for r in df.itertuples()
]

# Costruiamo l'indice FAISS (calcola gli embeddings di tutti i documenti).
vectorstore = FAISS.from_documents(documenti, embeddings)

# Il retriever restituirà i 4 documenti più pertinenti a ogni domanda.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Vector store FAISS pronto con {len(documenti)} documenti ✅")

> ℹ️ **Dettaglio su e5 e i prefissi.** Nella Lezione 2 abbiamo visto che la famiglia e5
> preferisce i prefissi `query:` (per la domanda) e `passage:` (per i documenti). Qui, per
> tenere la demo semplice, non li aggiungiamo: grazie a `normalize_embeddings=True` la
> ricerca per similarità funziona comunque bene per i nostri scopi. In un sistema di
> produzione si imposterebbero i prefissi (LangChain lo permette via
> `query_instruction` / `embed_instruction`) per spremere l'ultimo po' di qualità.

### Proviamo il solo retrieval 🔎

Prima di coinvolgere l'LLM, verifichiamo che il **retrieve** funzioni: data una domanda,
quali recensioni recupera? Sono queste le "fonti" che daremo in pasto al modello.

In [ ]:
domanda_test = "Cosa lamentano i clienti riguardo alla spedizione?"
fonti = retriever.invoke(domanda_test)

print(f"Domanda: {domanda_test}")
print(f"Documenti recuperati: {len(fonti)}\n")
for d in fonti:
    print(f"[{d.metadata['id']}] {d.metadata['prodotto']} ({d.metadata['rating']}★)")
    print(f"    {d.page_content}\n")

## (C) L'LLM Qwen, avvolto in LangChain 🤖

È la parte **`generate`**. Carichiamo lo stesso **`Qwen/Qwen2.5-7B-Instruct`** della
Lezione 4, in **4-bit** per farlo stare nella T4 (`BitsAndBytesConfig`). La novità rispetto
alla L4 è che, invece di chiamare `model.generate(...)` a mano, lo **avvolgiamo nei
componenti LangChain** (L5):

- **`HuggingFacePipeline`** — adatta una `pipeline` di Hugging Face all'interfaccia LangChain;
- **`ChatHuggingFace`** — aggiunge sopra il **chat template** (ruoli system/user/assistant),
  così possiamo usare i `ChatPromptTemplate` come nella Lezione 5.

> ⏳ La **prima** esecuzione **scarica** il modello (qualche GB): **1–3 minuti**. È il
> momento giusto per una pausa se stai registrando.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_id = "Qwen/Qwen2.5-7B-Instruct"

# Quantizzazione 4-bit (vedi L4): ~14 GB -> ~5-6 GB, entra nella T4.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")

# do_sample=False -> output deterministico (ideale per il RAG: vogliamo fedeltà al contesto).
# return_full_text=False -> la pipeline restituisce SOLO il testo generato, non il prompt.
gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
)

print("Modello caricato ✅  Memoria GPU usata: "
      f"{torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

# Avvolgiamo la pipeline come "chat model" LangChain (gestisce il chat template di Qwen).
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen))
print("LLM pronto per LangChain ✅")

> 💡 **Alternativa più leggera/veloce.** Se la T4 ti sembra lenta (o vai in *out of
> memory*), cambia `model_id = "Qwen/Qwen2.5-3B-Instruct"` nella cella sopra e riesegui:
> risponde più in fretta, con qualità un po' inferiore ma adeguata per la demo.

## (D) La catena RAG, con citazioni delle fonti 🔗

Ci siamo: colleghiamo i pezzi in un'unica **catena LCEL** (L5). Analizziamo i componenti.

**1) `formatta_contesto(docs)`** — prende i documenti recuperati dal retriever e li
trasforma in **un unico testo** da inserire nel prompt. Per ogni documento scriviamo l'**ID
tra parentesi quadre**, il prodotto e il rating: così l'LLM "vede" gli ID e può **citarli**
nella risposta.

**2) `prompt_rag`** — un `ChatPromptTemplate` con due messaggi:
- il `system` definisce le **regole d'oro del RAG**: rispondi in italiano, usa **SOLO** il
  contesto, **cita gli ID** tra parentesi quadre e, se l'informazione non c'è, **dillo**
  (è così che riduciamo le allucinazioni);
- l'`human` contiene i due "buchi" da riempire: `{contesto}` e `{domanda}`.

**3) `catena_rag`** — la composizione con l'operatore `|` (pipe) di LCEL:
- il dizionario iniziale prepara i due input del prompt **in parallelo**:
  `"contesto"` = `retriever | formatta_contesto` (recupera i doc e li formatta),
  `"domanda"` = `RunnablePassthrough()` (lascia passare la domanda così com'è);
- poi il flusso prosegue: `prompt_rag` → `chat` (l'LLM) → `StrOutputParser()` (estrae la
  stringa di testo dalla risposta).

Risultato: `catena_rag.invoke("...")` esegue **retrieve → generate** in un colpo solo.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def formatta_contesto(docs):
    # Un blocco di testo con un documento per riga, preceduto dal suo ID (per le citazioni).
    return "\n\n".join(
        f"[{d.metadata['id']}] {d.metadata['prodotto']} ({d.metadata['rating']}★): {d.page_content}"
        for d in docs
    )

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", "Sei un assistente che analizza recensioni clienti. Rispondi in italiano "
               "usando SOLO le informazioni nel contesto. Cita gli ID delle recensioni tra "
               "parentesi quadre. Se l'informazione non c'è nel contesto, dillo chiaramente."),
    ("human", "Contesto:\n{contesto}\n\nDomanda: {domanda}"),
])

catena_rag = (
    {"contesto": retriever | formatta_contesto, "domanda": RunnablePassthrough()}
    | prompt_rag
    | chat
    | StrOutputParser()
)

print(catena_rag.invoke("Cosa lamentano i clienti riguardo alla spedizione?"))

### Altre domande al sistema 💬

Proviamo la catena con altre domande realistiche, del tipo che porrebbe un *product
manager* davanti a migliaia di recensioni. Nota come la risposta **cita gli ID** delle
recensioni usate.

In [ ]:
print("=== Quali sono i punti di forza più citati? ===")
print(catena_rag.invoke("Quali sono i punti di forza più citati dai clienti?"))

In [ ]:
print("=== Riassumi le critiche sul prezzo ===")
print(catena_rag.invoke("Riassumi le critiche dei clienti sul prezzo dei prodotti."))

### Le fonti dietro la risposta 📎

Una risposta RAG è **affidabile** solo se possiamo verificarla. Mostriamo
**esplicitamente** quali documenti il retriever ha recuperato per una domanda: sono le
"fonti" che l'LLM ha letto. Confrontando gli **ID** citati nella risposta con questo
elenco, possiamo controllare che il modello non abbia inventato.

In [ ]:
domanda = "Riassumi le critiche dei clienti sul prezzo dei prodotti."
fonti = retriever.invoke(domanda)

print(f"Domanda: {domanda}")
print(f"Fonti recuperate ({len(fonti)} documenti):\n")
for d in fonti:
    print(f"[{d.metadata['id']}] {d.metadata['prodotto']} ({d.metadata['rating']}★)")
    print(f"    {d.page_content}\n")

## (E) Il cruscotto: la vista "business" 📊

Il RAG risponde a domande *in linguaggio naturale*. Ma un responsabile di prodotto vuole
spesso anche una **vista d'insieme quantitativa**. Con `pandas`, raggruppiamo le recensioni
per **prodotto** e calcoliamo tre indicatori chiave:
- **numero di recensioni**;
- **media del rating** (le stelle);
- **% di sentiment negativo** (la colonna `sentiment` arricchita nella sezione A).

È l'output "da dashboard" della nostra app: a colpo d'occhio si vedono i prodotti
problematici (poche stelle e molto sentiment negativo).

In [ ]:
cruscotto = df.groupby("prodotto").agg(
    n_recensioni=("id", "count"),
    media_rating=("rating", "mean"),
    perc_negativo=("sentiment", lambda s: (s == "negative").mean() * 100),
).round(1)

# Ordiniamo dai prodotti più critici (più sentiment negativo) ai migliori.
cruscotto = cruscotto.sort_values("perc_negativo", ascending=False)

print("CRUSCOTTO — sintesi per prodotto (ordinato per criticità):\n")
cruscotto

> 🔗 **Quadro completo.** Mettendo insieme le due viste otteniamo lo strumento ideale: dal
> **cruscotto** individuiamo *quale* prodotto ha problemi (es. alto % negativo), poi
> usiamo il **RAG** per chiedere *perché* ("Cosa non va nel prodotto X?") e leggere le
> motivazioni citate dalle recensioni. Quantità **+** spiegazione.

## Esercizio 🏋️

Tocca a te interrogare il sistema. Scegli **una** delle due tracce.

**A) La tua domanda.** Poni una tua domanda al RAG con `catena_rag.invoke("...")` e poi
mostra le fonti con `retriever.invoke("...")` per verificare che la risposta sia fondata.

**B) Un retriever filtrato.** A volte vogliamo restringere la ricerca a un sottoinsieme,
ad esempio **solo le recensioni negative di un certo prodotto**. FAISS supporta i **filtri
sui metadati**: basta passare `filter={...}` alle `search_kwargs`. Costruisci un retriever
filtrato, montaci sopra una catena RAG e interrogalo.

Completa il codice dove indicato dai **TODO**, poi esegui. Sotto trovi una **soluzione**.

In [ ]:
# --- TRACCIA A: la tua domanda ---
# TODO: scrivi la tua domanda e decommenta le due righe.
# mia_domanda = "..."
# print(catena_rag.invoke(mia_domanda))
# for d in retriever.invoke(mia_domanda):
#     print(f"[{d.metadata['id']}] {d.metadata['prodotto']} ({d.metadata['rating']}★)")

# --- TRACCIA B: retriever filtrato (scheletro) ---
# prodotto_target = "Smartwatch FitPro 2"
# retriever_filtrato = vectorstore.as_retriever(
#     search_kwargs={"k": 4, "filter": {"prodotto": prodotto_target}}  # TODO: prova anche {"sentiment": ...}
# )
# TODO: costruisci una catena RAG che usa 'retriever_filtrato' al posto di 'retriever'.

### ✅ Soluzione

La traccia **B** è la più istruttiva: un retriever filtrato per prodotto.

> ⚠️ **Attenzione al metadato.** Possiamo filtrare **solo sui campi che abbiamo messo nei
> metadati** del `Document` (sezione B): `id`, `prodotto`, `rating`. Per filtrare anche per
> `sentiment` dovremmo prima **aggiungerlo ai metadati** quando costruiamo i `Document`
> (lo facciamo nella cella seguente, ricostruendo il vector store).

In [ ]:
# Ricostruiamo i Document includendo ANCHE il sentiment nei metadati,
# così possiamo filtrare sia per prodotto sia per sentiment.
documenti_full = [
    Document(
        page_content=r.testo,
        metadata={"id": int(r.id), "prodotto": r.prodotto,
                  "rating": int(r.rating), "sentiment": r.sentiment},
    )
    for r in df.itertuples()
]
vectorstore_full = FAISS.from_documents(documenti_full, embeddings)

# Retriever filtrato: solo recensioni NEGATIVE di un prodotto specifico.
prodotto_target = "Smartwatch FitPro 2"
retriever_filtrato = vectorstore_full.as_retriever(
    search_kwargs={"k": 4, "filter": {"prodotto": prodotto_target, "sentiment": "negative"}}
)

# Catena RAG identica a prima, ma sul retriever filtrato.
catena_filtrata = (
    {"contesto": retriever_filtrato | formatta_contesto, "domanda": RunnablePassthrough()}
    | prompt_rag
    | chat
    | StrOutputParser()
)

domanda_es = f"Quali problemi segnalano i clienti su '{prodotto_target}'?"
print(catena_filtrata.invoke(domanda_es))

print("\n--- Fonti usate (solo recensioni negative del prodotto) ---")
for d in retriever_filtrato.invoke(domanda_es):
    print(f"[{d.metadata['id']}] {d.metadata['prodotto']} "
          f"({d.metadata['rating']}★, {d.metadata['sentiment']})")

## 🎓 Fine del corso — riepilogo, limiti e prossimi passi

**Ce l'abbiamo fatta!** Partendo da zero abbiamo costruito, pezzo per pezzo, una vera
applicazione di NLP & Generative AI. Guardiamo indietro al percorso completo:

| Lezione | Mattone | Confluito nel progetto come… |
|---|---|---|
| **L1** | Setup + prima `pipeline` | l'ambiente Colab/GPU su cui gira tutto |
| **L2** | Embeddings & ricerca semantica | il **retrieval** (FAISS) |
| **L3** | Modelli task-specific | l'**arricchimento** (sentiment) |
| **L4** | LLM generativi (Qwen 4-bit) | il **generate** del RAG |
| **L5** | LangChain (prompt, LCEL) | l'**orchestrazione** della catena |
| **L6** | **RAG** | l'**app completa** con citazioni + cruscotto |

### I limiti (importante esserne consapevoli) ⚠️
- **Allucinazioni:** il RAG le *riduce* molto, ma non le elimina. L'LLM può comunque
  interpretare male il contesto o citare ID sbagliati. Le **citazioni** servono proprio a
  permettere la verifica.
- **Qualità del retrieval:** se la ricerca recupera i documenti sbagliati, la risposta sarà
  sbagliata (*garbage in, garbage out*). I prefissi e5, un `k` ben scelto e un buon modello
  di embedding contano molto.
- **Modelli piccoli:** un 7B in 4-bit sulla T4 è notevole, ma resta più limitato dei grandi
  modelli commerciali; su domande complesse può perdere dettagli.

### Prossimi passi 🚀
- **Modelli più grandi/migliori** (o re-ranker del retrieval) per alzare la qualità.
- **Valutazione sistematica:** misurare *quanto* le risposte sono corrette e fondate (con
  set di domande di test, metriche di *faithfulness*).
- **Deploy:** trasformare il notebook in un servizio (API + interfaccia), con un vector
  store persistente e l'aggiornamento incrementale dell'archivio.

### Grazie! 🙏
Grazie di cuore per aver seguito tutte e sei le lezioni. Hai imparato a **usare** modelli
open source per capire il linguaggio, a far **generare** testo a un LLM e a **orchestrare**
il tutto in un'applicazione RAG che risponde citando le fonti. Sono fondamenta solide:
ora prendi questo progetto, applicalo ai **tuoi** dati e continua a costruire. 💪

📦 Tutto il materiale del corso: https://github.com/ccasadei-maggioli/corso-nlp-genai-2026